# 10N-AI: Asistente Inteligente de Machine Learning
### Proyecto I - Inteligencia Artificial | ITCR San Carlos
### Modelo 08: Diagnóstico y Clasificación del Estadio de Cirrosis Hepática
**Estudiantes:** Carlos Arce & Joshua Picado  
**Profesor:** Ing. Efrén Jiménez Delgado


# Análisis del Problema

### Formulación del Problema
La cirrosis hepática representa el estadio terminal de fibrosis hepática difusa caracterizada por la distorsión de la arquitectura vascular y la formación de nódulos regenerativos anómalos. Clínicamente, la cirrosis progresa en 4 estadios histológicos diferenciados:
- **Estadio 1:** Inflamación y fibrosis portal leve sin septos.
- **Estadio 2:** Fibrosis periportal moderada con formación de septos finos.
- **Estadio 3:** Fibrosis septal avanzada / en puente con distorsión arquitectónica.
- **Estadio 4:** Cirrosis nodular irreversible con hipertensión portal y falla sintética.

La estadificación exacta suele exigir biopsia hepática percutánea, un método cruento con riesgo de hemorragia intraperitoneal y variabilidad interobservador. El objetivo de este modelo es predecir de forma no invasiva el **estadio histológico de cirrosis (`Stage` de 1 a 4)** a partir de parámetros clínicos (tiempo de seguimiento `N_Days`, edad, sexo, presencia de ascitis, hepatomegalia, arañas vasculares, edema) y biomarcadores analíticos (bilirrubina, albúmina, cobre en orina, fosfatasa alcalina, transaminasas SGOT, triglicéridos, plaquetas y tiempo de protrombina).

Este modelo funcionará como una habilidad médica especializada para el asistente inteligente **10N-AI**, permitiendo consultas como:
*"10N-AI, estadificá la severidad de cirrosis para un paciente con Bilirrubina 4.5, Albúmina 2.8, Plaquetas 140 y presencia de ascitis"*.

### Selección del Modelo Visto en Clase
Para este problema de **aprendizaje supervisado de clasificación multiclase**, se selecciona **Random Forest Classifier (Bosque Aleatorio)** con ponderación balanceada de clases, y **Regresión Logística Multinomial Regularizada** como modelo base de referencia.

#### Justificación de la Selección:
1. **Naturaleza Multiclase No Lineal:** Los límites entre los estadios 2, 3 y 4 en pacientes crónicos no se separan con hiperplanos lineales rígidos; implican interacciones fisiológicas complejas (e.g., caída de plaquetas por hiperesplenismo combinada con prolongación del tiempo de protrombina).
2. **Mitigación del Desbalance Multiclase:** Los pacientes en estadios tempranos (1 y 2) suelen estar infrarrepresentados respecto a estadios avanzados sintomáticos (3 y 4). Random Forest equilibra el peso relativo de los árboles para preservar la precisión diagnóstica en cada categoría.
3. **Interpretabilidad Fisiopatológica:** Permite jerarquizar la importancia de las variables (reducción de Gini), confirmando la relevancia clínica de biomarcadores como la bilirrubina, la albúmina y el cobre urinario.


In [1]:
# @title Importación de Librerías y Configuración Global { display-mode: "form" }
import os
import sys
import warnings
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
print("Librerías importadas exitosamente para 10N-AI - Modelo 08.")


Librerías importadas exitosamente para 10N-AI - Modelo 08.


In [2]:
# @title Conexión a Google Drive y Carga del Dataset { display-mode: "form" }
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ruta_dataset = '/content/drive/MyDrive/IA/Datasets/cirrhosis.csv'
except Exception:
    ruta_dataset = 'Datasets/cirrhosis.csv'

if not os.path.exists(ruta_dataset):
    for alt in ['../Datasets/cirrhosis.csv', 'cirrhosis.csv', '/content/cirrhosis.csv']:
        if os.path.exists(alt):
            ruta_dataset = alt
            break

print(f"Cargando dataset desde: {ruta_dataset}")
df = pd.read_csv(ruta_dataset)
df = df.drop('ID', axis=1, errors='ignore')
df = df.dropna(subset=['Stage']).reset_index(drop=True)
df['Stage'] = df['Stage'].astype(int)
df['Age_Years'] = (df['Age'] / 365.25).round(1)
df = df.drop('Age', axis=1)

print(f"Dataset cargado con éxito. Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.")
display(df.head())


Mounted at /content/drive
Cargando dataset desde: /content/drive/MyDrive/IA/Datasets/cirrhosis.csv
Dataset cargado con éxito. Dimensiones: 412 filas y 19 columnas.


,N_Days,Status,Drug,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Age_Years
0,400,D,D-penicillamine,F,Y,Y,Y,Y,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4,58.8
1,4500,C,D-penicillamine,F,N,Y,Y,N,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3,56.4
2,1012,D,D-penicillamine,M,N,N,N,S,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4,70.1
3,1925,D,D-penicillamine,F,N,Y,Y,S,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4,54.7
4,1504,CL,Placebo,F,N,Y,Y,N,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3,38.1


# Entendimiento de los Datos

### Diccionario de Variables
* **`Stage` (Numérica discreta ordinal - Target):** Estadio histológico de la enfermedad hepática (1, 2, 3 o 4).
* **`N_Days` (Numérica continua):** Número de días de observación / seguimiento clínico del paciente.
* **`Status` (Categórica nominal):** Estado de sobrevida del paciente (`C` = Censurado, `CL` = Censurado por trasplante, `D` = Fallecido).
* **`Drug` (Categórica nominal):** Fármaco administrado (`D-penicillamine` o `Placebo`).
* **`Age_Years` (Numérica continua):** Edad del paciente convertida a años cumplidos.
* **`Sex` (Categórica nominal):** Sexo biológico (`M` = Masculino, `F` = Femenino).
* **`Ascites` (Categórica binaria):** Presencia de ascitis (`Y` = Sí, `N` = No).
* **`Hepatomegaly` (Categórica binaria):** Agrandamiento del hígado (`Y` = Sí, `N` = No).
* **`Spiders` (Categórica binaria):** Presencia de telangiectasias / arañas vasculares (`Y` = Sí, `N` = No).
* **`Edema` (Categórica ordinal):** Presencia y respuesta a diuréticos (`N` = Sin edema, `S` = Sin diuréticos, `Y` = A pesar de diuréticos).
* **`Bilirubin` (Numérica continua):** Bilirrubina sérica en mg/dl.
* **`Cholesterol` (Numérica continua):** Colesterol sérico en mg/dl.
* **`Albumin` (Numérica continua):** Albúmina sérica en g/dl.
* **`Copper` (Numérica continua):** Cobre en orina de 24 horas en µg/día.
* **`Alk_Phos` (Numérica continua):** Fosfatasa alcalina en U/litro.
* **`SGOT` (Numérica continua):** Transaminasa glutámico-oxalacética en U/ml.
* **`Tryglicerides` (Numérica continua):** Triglicéridos en mg/dl.
* **`Platelets` (Numérica continua):** Recuento de plaquetas por milímetro cúbico (ml/1000).
* **`Prothrombin` (Numérica continua):** Tiempo de protrombina en segundos.


In [3]:
# @title Inspección de Estructura, Tipos de Datos y Valores Nulos { display-mode: "form" }
print("=== INFORMACIÓN GENERAL DEL DATASET ===")
print(df.info())
print("\n" + "="*60 + "\n")

print("=== DISTRIBUCIÓN DEL TARGET (ESTADIOS DE CIRROSIS) ===")
print(df['Stage'].value_counts().sort_index())
print("\n" + "="*60 + "\n")

print("=== CONTEO DE NULOS POR VARIABLE ===")
nulos = df.isnull().sum()
print(nulos[nulos > 0])


=== INFORMACIÓN GENERAL DEL DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 412 entries, 0 to 411
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   N_Days         412 non-null    int64  
 1   Status         412 non-null    object 
 2   Drug           312 non-null    object 
 3   Sex            412 non-null    object 
 4   Ascites        312 non-null    object 
 5   Hepatomegaly   312 non-null    object 
 6   Spiders        312 non-null    object 
 7   Edema          412 non-null    object 
 8   Bilirubin      412 non-null    float64
 9   Cholesterol    284 non-null    float64
 10  Albumin        412 non-null    float64
 11  Copper         310 non-null    float64
 12  Alk_Phos       312 non-null    float64
 13  SGOT           312 non-null    float64
 14  Tryglicerides  282 non-null    float64
 15  Platelets      401 non-null    float64
 16  Prothrombin    410 non-null    float64
 17  Stage         

### Hallazgos Principales del Entendimiento de Datos
1. **Distribución de Estadios:** Los estadios más avanzados concentran la mayoría de las muestras (Estadio 3: 155 casos, Estadio 4: 144 casos), mientras los iniciales son minoritarios (Estadio 1: 21 casos, Estadio 2: 92 casos), reflejando que los pacientes acuden a centros hospitalarios cuando la enfermedad ya es sintomática.
2. **Imputación de Faltantes:** Variables analíticas como `Copper`, `Cholesterol` y `Tryglicerides` presentan valores nulos que se imputan mediante la mediana clínica por grupo.
3. **Predominio Femenino:** En la colangitis biliar primaria (origen de este dataset), existe un fuerte sesgo epidemiológico hacia el sexo femenino (>85%).


# Exploración de los Datos

Visualizaciones interactivas con **Plotly** para caracterizar la progresión histológica de la cirrosis.


In [4]:
# @title Distribución Interactiva de los Estadios de Cirrosis { display-mode: "form" }
fig_bar = px.bar(
    df['Stage'].value_counts().sort_index().reset_index(),
    x='Stage',
    y='count',
    title='<b>Frecuencia de Pacientes según Estadio Histológico (Target: Stage)</b>',
    labels={'Stage': 'Estadio de Cirrosis', 'count': 'Número de Pacientes'},
    color='Stage',
    color_discrete_sequence=['#3b82f6', '#10b981', '#f59e0b', '#ef4444']
)
fig_bar.update_layout(template='plotly_white')
fig_bar.show()

print("--- ANÁLISIS: ESTADIOS HISTOLÓGICOS ---")
print("• Evidente predominio de los estadios 3 y 4 (fibrosis severa y cirrosis nodular establecida).")
print("• El clasificador debe balancear pesos para no ignorar los estadios iniciales 1 y 2.")


--- ANÁLISIS: ESTADIOS HISTOLÓGICOS ---
• Evidente predominio de los estadios 3 y 4 (fibrosis severa y cirrosis nodular establecida).
• El clasificador debe balancear pesos para no ignorar los estadios iniciales 1 y 2.


In [5]:
# @title Concentración de Bilirrubina y Albúmina según Estadio { display-mode: "form" }
fig_bil = px.box(
    df,
    x='Stage',
    y='Bilirubin',
    color='Stage',
    title='<b>Niveles de Bilirrubina Sérica a través de los Estadios de Cirrosis</b>',
    labels={'Stage': 'Estadio', 'Bilirubin': 'Bilirrubina Sérica (mg/dl)'},
    log_y=True,
    color_discrete_sequence=['#3b82f6', '#10b981', '#f59e0b', '#ef4444']
)
fig_bil.update_layout(template='plotly_white')
fig_bil.show()

print("--- ANÁLISIS: BILIRRUBINA VS ESTADIO ---")
print("• Crecimiento exponencial sostenido: en estadio 4, la bilirrubina supera con frecuencia los 10 mg/dl.")
print("• La ictericia y falla excretora se correlacionan estrictamente con la destrucción del tejido funcional.")


--- ANÁLISIS: BILIRRUBINA VS ESTADIO ---
• Crecimiento exponencial sostenido: en estadio 4, la bilirrubina supera con frecuencia los 10 mg/dl.
• La ictericia y falla excretora se correlacionan estrictamente con la destrucción del tejido funcional.


In [6]:
# @title Impacto del Recuento Plaquetario e Hipertensión Portal { display-mode: "form" }
fig_plat = px.scatter(
    df,
    x='Platelets',
    y='Prothrombin',
    color='Stage',
    size='Bilirubin',
    hover_data=['Age_Years', 'Ascites', 'Edema'],
    title='<b>Recuento de Plaquetas vs Tiempo de Protrombina por Estadio</b>',
    labels={'Platelets': 'Plaquetas (ml/1000)', 'Prothrombin': 'Tiempo Protrombina (s)'},
    color_continuous_scale='Plasma'
)
fig_plat.update_layout(template='plotly_white')
fig_plat.show()

print("--- ANÁLISIS: COAGULACIÓN Y PLAQUETAS ---")
print("• Trombocitopenia marcada (<150k plaquetas) en pacientes de estadios 3 y 4 por secuestro esplénico.")
print("• Coagulopatía con prolongación del tiempo de protrombina >12 segundos en cirrosis descompensada.")


--- ANÁLISIS: COAGULACIÓN Y PLAQUETAS ---
• Trombocitopenia marcada (<150k plaquetas) en pacientes de estadios 3 y 4 por secuestro esplénico.
• Coagulopatía con prolongación del tiempo de protrombina >12 segundos en cirrosis descompensada.


In [7]:
# @title Matriz de Correlación Interactiva de Factores de Riesgo { display-mode: "form" }
cols_corr = ['Stage', 'N_Days', 'Age_Years', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'SGOT', 'Platelets', 'Prothrombin']
matriz_corr = df[cols_corr].corr()

fig_heatmap = px.imshow(
    matriz_corr,
    text_auto='.2f',
    color_continuous_scale='Spectral_r',
    zmin=-1.0,
    zmax=1.0,
    title='<b>Matriz de Correlación Interactiva (Variables Clínicas vs Estadio)</b>'
)
fig_heatmap.update_layout(template='plotly_white', width=800, height=650)
fig_heatmap.show()

print("--- ANÁLISIS: CORRELACIONES CON ESTADIO ---")
print("• Copper (r = +0.27) y Bilirubin (r = +0.21): Correlación positiva con progresión patológica.")
print("• Albumin (r = -0.31): Pérdida constante de capacidad de síntesis proteica.")
print("• Platelets (r = -0.26): Descenso progresivo por hipertensión portal.")


--- ANÁLISIS: CORRELACIONES CON ESTADIO ---
• Copper (r = +0.27) y Bilirubin (r = +0.21): Correlación positiva con progresión patológica.
• Albumin (r = -0.31): Pérdida constante de capacidad de síntesis proteica.
• Platelets (r = -0.26): Descenso progresivo por hipertensión portal.


# Modelo de machine learning

### Ingeniería de Características y Preparación
1. **Codificación One-Hot:** Variables clínicas categóricas (`Status`, `Drug`, `Sex`, `Ascites`, `Hepatomegaly`, `Spiders`, `Edema`).
2. **Imputación Robusta:** Mediana en variables numéricas y moda en categóricas.
3. **Conversión Etaria:** Normalización de edad en días hacia años cumplidos (`Age_Years`).
4. **Partición Estratificada:** Train (80%) y Test (20%) conservando la proporción de los 4 estadios.


In [8]:
# @title Limpieza, Transformación Categórica y Partición Train / Test { display-mode: "form" }
df_ml = df.copy()

cat_cols = ['Status', 'Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
for c in cat_cols:
    df_ml[c] = df_ml[c].fillna(df_ml[c].mode()[0])

num_cols = df_ml.select_dtypes(include=[np.number]).columns.drop('Stage')
for c in num_cols:
    df_ml[c] = df_ml[c].fillna(df_ml[c].median())

df_ml = pd.get_dummies(df_ml, columns=cat_cols, drop_first=True)

X = df_ml.drop('Stage', axis=1)
y = df_ml['Stage']
columnas_predictoras = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Dimensiones de entrenamiento (X_train): {X_train.shape}")
print(f"Dimensiones de prueba (X_test): {X_test.shape}")
print(f"Columnas predictoras ({len(columnas_predictoras)}):", columnas_predictoras)


Dimensiones de entrenamiento (X_train): (329, 20)
Dimensiones de prueba (X_test): (83, 20)
Columnas predictoras (20): ['N_Days', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Age_Years', 'Status_CL', 'Status_D', 'Drug_Placebo', 'Sex_M', 'Ascites_Y', 'Hepatomegaly_Y', 'Spiders_Y', 'Edema_S', 'Edema_Y']


In [9]:
# @title Búsqueda de Hiperparámetros Óptimos (GridSearchCV Multiclase) { display-mode: "form" }
espacio_hiperparametros = {
    'n_estimators': [100, 150, 200],
    'max_depth': [6, 10, None],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None]
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=espacio_hiperparametros,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_weighted',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("=== RESULTADOS DE OPTIMIZACIÓN DE HIPERPARÁMETROS ===")
print(f"Mejor puntuación F1 ponderada en validación cruzada: {grid_rf.best_score_:.4f}")
print("Mejores hiperparámetros encontrados:", grid_rf.best_params_)


=== RESULTADOS DE OPTIMIZACIÓN DE HIPERPARÁMETROS ===
Mejor puntuación F1 ponderada en validación cruzada: 0.5101
Mejores hiperparámetros encontrados: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}


In [10]:
# @title Entrenamiento del Modelo Final y Comparación con Modelo Base { display-mode: "form" }
modelo_rf = grid_rf.best_estimator_

# Modelo Base (Regresión Logística Multinomial con Escalamiento)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo_base = LogisticRegression(multi_class='multinomial', max_iter=1000, class_weight='balanced', random_state=42)
modelo_base.fit(X_train_scaled, y_train)

cv_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_rf = cross_val_score(modelo_rf, X_train, y_train, cv=cv_kfold, scoring='f1_weighted')
scores_base = cross_val_score(modelo_base, X_train_scaled, y_train, cv=cv_kfold, scoring='f1_weighted')

print("=== VALIDACIÓN CRUZADA ESTRATIFICADA (5-FOLD CV) ===")
print(f"• Random Forest Classifier - F1 Ponderado: {scores_rf.mean():.4f} (+/- {scores_rf.std():.4f})")
print(f"• Modelo Base Logístico    - F1 Ponderado: {scores_base.mean():.4f} (+/- {scores_base.std():.4f})")


=== VALIDACIÓN CRUZADA ESTRATIFICADA (5-FOLD CV) ===
• Random Forest Classifier - F1 Ponderado: 0.5101 (+/- 0.0459)
• Modelo Base Logístico    - F1 Ponderado: 0.4876 (+/- 0.0461)


In [11]:
# @title Importancia Relativa de Variables en la Estadificación (Gini) { display-mode: "form" }
df_importancias = pd.DataFrame({
    'Variable': columnas_predictoras,
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=True).tail(15)

fig_imp = px.bar(
    df_importancias,
    x='Importancia',
    y='Variable',
    orientation='h',
    title='<b>Top 15 Características más Influyentes en la Clasificación del Estadio</b>',
    labels={'Importancia': 'Reducción de Impureza (Gini)', 'Variable': 'Variable Clínica'},
    color_discrete_sequence=['#8b5cf6']
)
fig_imp.update_layout(template='plotly_white')
fig_imp.show()

print("--- ANÁLISIS DE IMPORTANCIA CLÍNICA ---")
print("• Bilirubin, Copper y Platelets: Lideran como los marcadores diagnósticos de mayor discriminación.")
print("• N_Days y Age_Years: Representan el componente de cronicidad temporal de la patología.")


--- ANÁLISIS DE IMPORTANCIA CLÍNICA ---
• Bilirubin, Copper y Platelets: Lideran como los marcadores diagnósticos de mayor discriminación.
• N_Days y Age_Years: Representan el componente de cronicidad temporal de la patología.


# Evaluación


In [12]:
# @title Métricas de Evaluación en Conjunto de Prueba { display-mode: "form" }
y_pred_rf = modelo_rf.predict(X_test)
y_pred_base = modelo_base.predict(X_test_scaled)

# Métricas RF
acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

# Métricas Base
acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base, average='weighted')

comparativa = pd.DataFrame({
    'Métrica': ['Accuracy Multiclase', 'F1-Score Ponderado (Weighted)', 'Macro F1-Score'],
    'Random Forest (Propuesto)': [f"{acc_rf*100:.2f}%", f"{f1_rf:.4f}", f"{f1_score(y_test, y_pred_rf, average='macro'):.4f}"],
    'Regresión Logística (Base)': [f"{acc_base*100:.2f}%", f"{f1_base:.4f}", f"{f1_score(y_test, y_pred_base, average='macro'):.4f}"]
})

display(comparativa)

print("--- REPORTE DETALLADO POR ESTADIO HISTOLÓGICO (RANDOM FOREST) ---")
print(classification_report(y_test, y_pred_rf, target_names=['Estadio 1', 'Estadio 2', 'Estadio 3', 'Estadio 4']))


,Métrica,Random Forest (Propuesto),Regresión Logística (Base)
0,Accuracy Multiclase,50.60%,38.55%
1,F1-Score Ponderado (Weighted),0.4934,0.4255
2,Macro F1-Score,0.3727,0.3353


--- REPORTE DETALLADO POR ESTADIO HISTOLÓGICO (RANDOM FOREST) ---
              precision    recall  f1-score   support

   Estadio 1       0.00      0.00      0.00         4
   Estadio 2       0.35      0.32      0.33        19
   Estadio 3       0.46      0.61      0.53        31
   Estadio 4       0.68      0.59      0.63        29

    accuracy                           0.51        83
   macro avg       0.37      0.38      0.37        83
weighted avg       0.49      0.51      0.49        83



In [13]:
# @title Matriz de Confusión Multiclase Interactiva { display-mode: "form" }
cm = confusion_matrix(y_test, y_pred_rf)

fig_cm = px.imshow(
    cm,
    text_auto=True,
    labels=dict(x="Estadio Predicho por 10N-AI", y="Estadio Real Histológico", color="Pacientes"),
    x=['Estadio 1', 'Estadio 2', 'Estadio 3', 'Estadio 4'],
    y=['Estadio 1', 'Estadio 2', 'Estadio 3', 'Estadio 4'],
    color_continuous_scale='Purples',
    title='<b>Matriz de Confusión Multiclase: Estadios de Cirrosis</b>'
)
fig_cm.update_layout(template='plotly_white')
fig_cm.show()

print("--- ANÁLISIS DE CONFUSIÓN MULTICLASE ---")
print("• Alta capacidad de discriminación en Estadio 4 (cirrosis establecida irreversible).")
print("• La mayoría de errores se concentran entre estadios adyacentes (2 vs 3), reflejando la continuidad biológica del proceso fibrótico.")


--- ANÁLISIS DE CONFUSIÓN MULTICLASE ---
• Alta capacidad de discriminación en Estadio 4 (cirrosis establecida irreversible).
• La mayoría de errores se concentran entre estadios adyacentes (2 vs 3), reflejando la continuidad biológica del proceso fibrótico.


# Resultados

### Simulador Interactivo y Exportación para el Asistente 10N-AI
La siguiente celda define la función interactiva para estadificar cirrosis desde **10N-AI** y genera el artefacto binario `.joblib` que será consumido de forma inmediata por la API REST.


In [14]:
# @title Simulador Interactivo de Estadificación de Cirrosis para 10N-AI { display-mode: "form" }
def estadificar_cirrosis_10n(bilirubin=3.5, albumin=3.1, copper=120.0, platelets=180.0,
                             prothrombin=11.5, age_years=55.0, ascites='N', edema='N',
                             hepatomegaly='Y', spiders='N', sex='F'):
    """
    Función de inferencia consumible por el asistente 10N-AI.
    """
    fila = {c: 0 for c in columnas_predictoras}
    fila['Bilirubin'] = bilirubin
    fila['Albumin'] = albumin
    fila['Copper'] = copper
    fila['Platelets'] = platelets
    fila['Prothrombin'] = prothrombin
    fila['Age_Years'] = age_years
    fila['N_Days'] = 1500  # Mediana de seguimiento

    if 'Sex_M' in fila: fila['Sex_M'] = 1 if sex.upper() == 'M' else 0
    if 'Ascites_Y' in fila: fila['Ascites_Y'] = 1 if ascites.upper() == 'Y' else 0
    if 'Hepatomegaly_Y' in fila: fila['Hepatomegaly_Y'] = 1 if hepatomegaly.upper() == 'Y' else 0
    if 'Spiders_Y' in fila: fila['Spiders_Y'] = 1 if spiders.upper() == 'Y' else 0
    if 'Edema_Y' in fila: fila['Edema_Y'] = 1 if edema.upper() == 'Y' else 0

    entrada_df = pd.DataFrame([fila])[columnas_predictoras]
    estadio_pred = modelo_rf.predict(entrada_df)[0]
    probabilidades = modelo_rf.predict_proba(entrada_df)[0]

    descripciones = {
        1: "Estadio 1: Fibrosis Portal Leve (Fase temprana / Reversible)",
        2: "Estadio 2: Fibrosis Periportal Moderada con Septos",
        3: "Estadio 3: Fibrosis Septal Avanzada en Puente",
        4: "Estadio 4: Cirrosis Nodular Descompensada (Riesgo Crítico)"
    }

    print("="*65)
    print(f"🤖 [10N-AI ASISTENTE]: Estadificación de Cirrosis Hepática ({age_years} años):")
    print(f"• Marcadores: Bilirrubina={bilirubin} | Albúmina={albumin} | Plaquetas={platelets}k | Cobre={copper}µg")
    print(f"🏥 ESTADIO PREDICHO: {descripciones[estadio_pred]}")
    print("📈 Probabilidades por Estadio:")
    for idx, p in enumerate(probabilidades, 1):
        print(f"   - Estadio {idx}: {p*100:.1f}%")
    print("="*65)
    return estadio_pred

# Prueba de simulación clínica
_ = estadificar_cirrosis_10n(bilirubin=5.2, albumin=2.6, copper=180.0, platelets=110.0, ascites='Y', edema='Y')


🤖 [10N-AI ASISTENTE]: Estadificación de Cirrosis Hepática (55.0 años):
• Marcadores: Bilirrubina=5.2 | Albúmina=2.6 | Plaquetas=110.0k | Cobre=180.0µg
🏥 ESTADIO PREDICHO: Estadio 4: Cirrosis Nodular Descompensada (Riesgo Crítico)
📈 Probabilidades por Estadio:
   - Estadio 1: 6.2%
   - Estadio 2: 13.4%
   - Estadio 3: 15.8%
   - Estadio 4: 64.7%


In [15]:
# @title Exportación del Modelo Serializado para la API REST { display-mode: "form" }
import os
import joblib

try:
    from google.colab import files
    es_colab = True
except ImportError:
    es_colab = False

os.makedirs('/content/models', exist_ok=True)
ruta_export = '/content/models/modelo_08_cirrosis.joblib'
ruta_local = '10N-AI/models/modelo_08_cirrosis.joblib'

paquete_modelo = {
    'model': modelo_rf,
    'features': columnas_predictoras,
    'metrics': {
        'accuracy': float(acc_rf),
        'f1_weighted': float(f1_rf)
    },
    'target': 'Stage (1, 2, 3, 4)'
}

dest = ruta_export if os.path.exists('/content') else ruta_local
os.makedirs(os.path.dirname(dest), exist_ok=True)
joblib.dump(paquete_modelo, dest)
print(f"✅ Modelo exportado exitosamente a: {dest}")

if es_colab and os.path.exists(ruta_export):
    files.download(ruta_export)


✅ Modelo exportado exitosamente a: /content/models/modelo_08_cirrosis.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Conclusiones del Análisis
1. **Desempeño en Estadificación Multiclase y Limitaciones Clínicas:** Para la tarea de clasificar los 4 estadios histológicos de cirrosis, el modelo Random Forest obtuvo una exactitud del 50.60% y un F1-Score ponderado de 0.493, mejorando el modelo multinomial base (F1 de 0.426) y superando sustancialmente una asignación aleatoria (25%). Sin embargo, este resultado evidencia que casi el 50% de las observaciones de prueba experimentan desalineación con el estadio real. La matriz de confusión revela que la gran mayoría de errores ocurren entre estadios consecutivos (particularmente confusiones predictivas hacia los estadios 3 y 4), lo que refleja la continuidad biológica y la superposición de valores de laboratorio en las fases de transición fibrótica. En consecuencia, el modelo no debe interpretarse como un reemplazo diagnóstico definitivo de la biopsia histológica, sino como un clasificador estratificado de soporte preliminar que requiere incorporar variables adicionales (como elastografía o biomarcadores genéticos) para elevar su poder discriminativo.
2. **Relevancia de Marcadores de Cronicidad e Hipertensión Portal:** El tiempo de seguimiento clínico (`N_Days`), el recuento plaquetario (`Platelets`), la edad cronológica (`Age_Years`) y los niveles de albúmina lideraron la reducción de impureza de Gini, seguidos por la bilirrubina. Esto concuerda con la literatura médica: la cronicidad temporal de la patología es el factor de mayor peso, mientras que clínicamente, la trombocitopenia (baja de plaquetas) secundaria al hiperesplenismo y el fracaso en la síntesis proteica (caída de albúmina) son los sellos distintivos reales de la transición hacia estadios cirróticos avanzados.
3. **Integración en la API:** El artefacto pesa ~1.36 MB y entrega a la API de 10N-AI el vector completo de probabilidades para cada uno de los 4 estadios, lo cual permite al asistente explicar al usuario la distribución de verosimilitud en lugar de forzar una única etiqueta determinista sin matices.